# Qwen3-14B 감성분석 파인튜닝 - EDA

> 데이터셋: `Younggooo/senti_data2` (150건, 한국어 제품 리뷰)

**분석 항목:**
1. 데이터셋 기본 정보
2. 감성 분포
3. 토픽 빈도 분석
4. 텍스트 길이 분석 (max_seq_length 검증)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer
from dotenv import load_dotenv

load_dotenv("../.env")

plt.rcParams["font.family"] = "AppleGothic"  # macOS 한글 폰트
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

# 데이터셋 로드
ds = load_dataset("Younggooo/senti_data2", split="train")
df = ds.to_pandas()
print(f"데이터 수: {len(df)}")
print(f"컬럼: {list(df.columns)}")
df.head()

## 1. 기본 정보

In [ ]:
# 데이터 타입 및 결측치 확인
print("=== dtypes ===")
print(df.dtypes)
print(f"\n=== 결측치 ===")
print(df.isnull().sum())
print(f"\n=== 기본 통계 ===")
df.describe(include="all")

## 2. 감성 분포

In [ ]:
# 감성 라벨 분포
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 막대 차트
sentiment_counts = df["sentiment"].value_counts()
colors = {"positive": "#4CAF50", "negative": "#F44336", "neutral": "#FFC107"}
sentiment_counts.plot(
    kind="bar",
    ax=axes[0],
    color=[colors.get(s, "#999") for s in sentiment_counts.index],
)
axes[0].set_title("감성 분포 (건수)")
axes[0].set_ylabel("건수")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

for i, (label, count) in enumerate(sentiment_counts.items()):
    axes[0].text(i, count + 0.5, str(count), ha="center", fontweight="bold")

# 파이 차트
sentiment_counts.plot(
    kind="pie",
    ax=axes[1],
    autopct="%1.1f%%",
    colors=[colors.get(s, "#999") for s in sentiment_counts.index],
)
axes[1].set_title("감성 분포 (비율)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

print(f"\n감성 분포:")
for label, count in sentiment_counts.items():
    print(f"  {label}: {count}건 ({count/len(df)*100:.1f}%)")

In [ ]:
# 확률값 분포
df["prob_float"] = df["probability"].astype(float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 전체 확률 분포
df["prob_float"].hist(bins=20, ax=axes[0], color="#5C6BC0", edgecolor="white")
axes[0].set_title("확률값 분포 (전체)")
axes[0].set_xlabel("probability")
axes[0].set_ylabel("건수")

# 감성별 확률 분포
for label, color in colors.items():
    subset = df[df["sentiment"] == label]["prob_float"]
    if len(subset) > 0:
        subset.hist(bins=15, ax=axes[1], alpha=0.6, label=label, color=color, edgecolor="white")
axes[1].set_title("확률값 분포 (감성별)")
axes[1].set_xlabel("probability")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"확률값 통계:")
print(df["prob_float"].describe())

## 3. 토픽 빈도 분석

In [ ]:
# 토픽 빈도 집계
def extract_topics(series):
    topics = []
    for val in series.dropna():
        if val.strip():
            topics.extend([t.strip() for t in val.split(",") if t.strip()])
    return Counter(topics)

pos_topics = extract_topics(df["positive_topics"])
neg_topics = extract_topics(df["negative_topics"])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 긍정 토픽 Top 15
if pos_topics:
    top_pos = pos_topics.most_common(15)
    axes[0].barh([t[0] for t in top_pos][::-1], [t[1] for t in top_pos][::-1], color="#4CAF50")
    axes[0].set_title(f"긍정 토픽 Top 15 (총 {len(pos_topics)}종)")
    axes[0].set_xlabel("빈도")

# 부정 토픽 Top 15
if neg_topics:
    top_neg = neg_topics.most_common(15)
    axes[1].barh([t[0] for t in top_neg][::-1], [t[1] for t in top_neg][::-1], color="#F44336")
    axes[1].set_title(f"부정 토픽 Top 15 (총 {len(neg_topics)}종)")
    axes[1].set_xlabel("빈도")

plt.tight_layout()
plt.show()

print(f"긍정 토픽 종류: {len(pos_topics)}개, 총 빈도: {sum(pos_topics.values())}")
print(f"부정 토픽 종류: {len(neg_topics)}개, 총 빈도: {sum(neg_topics.values())}")

## 4. 텍스트 길이 분석 (max_seq_length 검증)

In [ ]:
# 문자 수 분석
df["char_len"] = df["Input"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df["char_len"].hist(bins=30, ax=axes[0], color="#5C6BC0", edgecolor="white")
axes[0].set_title("입력 텍스트 길이 분포 (문자 수)")
axes[0].set_xlabel("문자 수")
axes[0].set_ylabel("건수")
axes[0].axvline(df["char_len"].mean(), color="red", linestyle="--", label=f'평균: {df["char_len"].mean():.0f}')
axes[0].legend()

# 감성별 길이 분포
df.boxplot(column="char_len", by="sentiment", ax=axes[1])
axes[1].set_title("감성별 텍스트 길이")
axes[1].set_ylabel("문자 수")
plt.suptitle("")

plt.tight_layout()
plt.show()

print(f"텍스트 길이 통계 (문자 수):")
print(df["char_len"].describe())

In [ ]:
# 토큰 수 분석 (Qwen3 토크나이저 기준)
# 토크나이저를 로드할 수 없는 환경에서는 이 셀을 건너뛰세요
try:
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-14B", trust_remote_code=True)

    df["token_len"] = df["Input"].apply(lambda x: len(tokenizer.encode(x)))

    fig, ax = plt.subplots(figsize=(8, 4))
    df["token_len"].hist(bins=30, ax=ax, color="#FF7043", edgecolor="white")
    ax.set_title("입력 텍스트 토큰 수 분포 (Qwen3 토크나이저)")
    ax.set_xlabel("토큰 수")
    ax.set_ylabel("건수")
    ax.axvline(512, color="red", linestyle="--", linewidth=2, label="max_seq_length=512")
    ax.axvline(df["token_len"].mean(), color="blue", linestyle="--", label=f'평균: {df["token_len"].mean():.0f}')
    ax.legend()
    plt.tight_layout()
    plt.show()

    over_512 = (df["token_len"] > 400).sum()  # 입력+프롬프트+출력 합산 여유분 고려
    print(f"토큰 수 통계:")
    print(df["token_len"].describe())
    print(f"\n입력 토큰 400 이상 (프롬프트+출력 포함 시 512 초과 가능): {over_512}건")
    print(f"max_seq_length=512 {'적절' if over_512 == 0 else '재검토 필요'}")
except Exception as e:
    print(f"토크나이저 로드 실패 (서버 환경에서 실행하세요): {e}")
    print("문자 수 기준으로 대략적 토큰 수를 추정합니다.")
    df["est_token_len"] = df["char_len"] * 0.5  # 한국어 대략 2자=1토큰
    print(f"추정 토큰 수 최대: {df['est_token_len'].max():.0f}")
    print(f"max_seq_length=512 충분할 것으로 예상")

## 5. 샘플 데이터 확인

In [ ]:
# 감성별 샘플 확인
for sentiment in ["positive", "negative", "neutral"]:
    subset = df[df["sentiment"] == sentiment]
    if len(subset) > 0:
        sample = subset.iloc[0]
        print(f"\n{'='*60}")
        print(f"[{sentiment.upper()}] probability: {sample['probability']}")
        print(f"Input: {sample['Input'][:200]}")
        print(f"positive_topics: {sample['positive_topics']}")
        print(f"negative_topics: {sample['negative_topics']}")